# Truncation on Real Logits: Does the Controlled Ordering Survive the Loss of Ground Truth?

**A research note on decoding operators applied to a real language model.** The previous study scored three truncation operators against a designed head and tail. Here the oracle disappears: on real SmolLM2 logits nobody knows which tokens *should* be removed. This study replaces ground truth with measurable proxies and asks whether the controlled ordering transfers to data.

---

## Abstract

This artifact measures top-k, top-p and min-p on real SmolLM2-135M next-token distributions, on CPU, fully reproducible. **What/how:** the cached logit tensor from the first study supplies real positions spanning `k90` from 1 to 12,097; each operator at its conventional default is applied to every position and scored on `n_kept`, `mass_kept`, `D_KL(p′‖p)` and `exp(H′)`, with the spread across positions replacing the oracle as the stability measure. Temperature is then composed with truncation in both orders, since the two do not commute. **What it shows:** the mass-based rule is still the most stable across positions, but three of six pre-registered predictions reverse — most sharply the claim that operator ordering matters less than operator choice. **What it does not claim:** no judgement of generated text quality. Correctness rests on the numbers the cells produce.

## Related work, and what changes here

| Ref | Work | Contribution | Where this study goes further |
|---|---|---|---|
| Holtzman et al. (2020) | *Neural Text Degeneration*, arXiv:1904.09751 | Nucleus sampling; the unreliable-tail argument | They report generation quality; we report the per-position distribution of what the cut removes |
| Fan et al. (2018) | *Hierarchical Neural Story Generation*, arXiv:1805.04833 | Top-k sampling | We measure what a fixed `k` means at positions whose true width ranges 1 to 12,097 |
| Nguyen et al. (2024) | *Min-p Sampling*, arXiv:2407.01082 | Relative-height truncation, claimed adaptive | The controlled study reversed that claim; this asks whether real logits agree |
| Meister et al. (2023) | *Locally Typical Sampling*, arXiv:2202.00666 | Information-content truncation | Frames why entropy-relative rules are the natural alternative to rank and mass |
| Zhu et al. (2024) | *Hot or Cold? Adaptive Temperature Sampling*, AAAI | Temperature as a per-position decision | Motivates the ordering experiment: `T` before or after the mask is not the same operator |
| Allal et al. (2025) | *SmolLM2*, arXiv:2502.02737 | The model under measurement | 49,152-way head on a 576-wide model |

**The differentiator.** The controlled study could score a cut because the head was designed. On real logits that is impossible, so the standard move is to evaluate downstream text quality. We take the other route: keep the measurement at the distribution level, and replace "was the cut correct" with "how much does the cut vary across positions the operator will actually meet".

## The measurement object and metrics

**Object.** The cached logit tensor `Z ∈ R^{83×49152}` from the first study, reused rather than recomputed — no model enters the kernel. Each row is a real next-token distribution `p = softmax(z)`, and the first study established their spread: median `top1` 0.259, median `exp(H)` 66.6, median `k90` 222, with `k90` ranging from **1** to **12,097**.

**What replaces the oracle.** Three proxies, none of which needs ground truth:

- **`n_kept` across positions** — the surviving branching factor. Its *spread* measures how much a fixed hyperparameter changes meaning as the distribution moves.
- **`mass_kept` across positions** — coverage. Pinned by construction for top-p, free for the other two.
- **`D_KL(p′‖p) = −log(mass_kept)`** — the distortion identity, applied to real rows.
- **`exp(H′)`** — the effective branching factor after the cut.

**Composition, and why order matters.** Temperature and truncation do not commute. For top-k the support is *rank*-defined and temperature is rank-preserving, so the two orders give the same support — but not the same probabilities. For top-p and min-p even the support changes.

## Hypothesis board (decided before measurement)

| # | Claim (falsifiable) | Predicted |
|---|---|---|
| C1 | A fixed `k = 50` means very different things across real positions | `mass_kept` spans > 0.5 across the 83 positions |
| C2 | Top-p's count varies more on real logits than in the controlled study | `n_kept` spread > 355× (the synthetic figure) |
| C3 | The controlled stability ordering transfers | `mass_kept` spread: top-p smallest, then min-p, then top-k |
| C4 | Min-p is most aggressive at high-entropy positions | min-p's `n_kept` correlates *negatively* with the position's `k90` |
| C5 | Operator ordering matters less than operator choice | ordering TVD < between-operator TVD |
| C6 | On the widest position, all three keep a small fraction of the vocabulary | every operator keeps < 25% of `V` at position 0 |

**How a verdict is won.** C1, C2, C3 are spread claims across the 83 real positions. C4 is a sign claim on a rank correlation. C5 compares two distances. C6 is a single-position check.

**Pre-committed honesty note.** Eighty-three positions from one document under one model, not independent. Nothing here says whether a cut was *correct* — only how much it varied and how far it moved the distribution.

## Protocol & constraints

- **No model load.** `pps_cache/logits_135M.pt` is read directly; if absent, the first study's rig regenerates it.
- **Precision:** float64 for the operator arithmetic, converted from the cached float32 logits.
- **Operators:** identical implementations to the previous study — same nucleus boundary convention (`+1`, boundary token included), same guards.
- **Defaults:** `k = 50`, `p_nuc = 0.9`, `α = 0.1`, matching the controlled study.
- **Determinism:** nothing is sampled. Every quantity is a deterministic function of the cached logits.

## The real-distribution rig

**Why:** the previous study's operators need real rows to act on, and the comparison between studies is only valid if the code is identical. This cell loads the cache, converts to float64, and re-establishes the per-position shape statistics.

**What the run shows:** the same 83 positions with the same medians measured earlier, now in float64, plus confirmation that the operator code reproduces the controlled study's behaviour on a real row.

In [1]:
import torch, math, numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

blob = torch.load(Path("pps_cache") / "logits_135M.pt", map_location="cpu", weights_only=True)
Z, ids = blob["logits"].double(), blob["ids"]
T, V = Z.shape
P = torch.softmax(Z, dim=-1)

H_all    = -(P * P.clamp_min(1e-300).log()).sum(-1)
srt      = P.sort(dim=-1, descending=True).values
k90_all  = (srt.cumsum(-1) < 0.90).sum(-1) + 1
top1_all = srt[:, 0]

print(f"positions T = {T}  vocab V = {V}  dtype {P.dtype}")
print(f"{'stat':10s} {'median':>10} {'min':>10} {'max':>10}")
for nm, v in [("top1", top1_all), ("H", H_all), ("exp(H)", H_all.exp()), ("k90", k90_all.double())]:
    print(f"{nm:10s} {v.median():10.3f} {v.min():10.3f} {v.max():10.3f}")
print(f"row sums ok: {bool((P.sum(-1) - 1).abs().max() < 1e-12)}")

def keep_topk(p, k):
    m = torch.zeros_like(p, dtype=torch.bool); m[p.topk(k).indices] = True; return m

def keep_topp(p, p_nuc):
    s, o = p.sort(descending=True)
    n = int((s.cumsum(0) < p_nuc).sum()) + 1
    m = torch.zeros_like(p, dtype=torch.bool); m[o[:n]] = True; return m

def keep_minp(p, alpha):
    return p >= alpha * p.max()

def renorm(p, m):
    q = torch.where(m, p, torch.zeros_like(p)); return q / q.sum()

def score_cut(p, m):
    assert bool(m.any())
    pp = renorm(p, m)
    Hp = float(-(pp[m] * pp[m].log()).sum())
    M  = float(p[m].sum())
    return {"n_kept": int(m.sum()), "mass_kept": M, "kl": -math.log(M), "expH": math.exp(Hp)}

OPS     = {"topk": keep_topk, "topp": keep_topp, "minp": keep_minp}
DEFAULT = {"topk": 50, "topp": 0.9, "minp": 0.1}
print("operators ready (identical to the controlled study)")

positions T = 83   vocab V = 49152   dtype torch.float64
stat           median        min        max
top1            0.259      0.022      0.995
H               4.198      0.053      8.647
exp(H)         66.570      1.054   5691.698
k90           222.000      1.000  12097.000
row sums ok: True
operators ready (identical to the controlled study)

## One fixed setting, eighty-three real distributions

**Why:** in deployment a hyperparameter is chosen once and then meets every position the text produces. The controlled study manufactured that variation by sweeping a Zipf exponent; here the document supplies it directly, and the range is wider than anything constructed.

**What the run shows.** Top-p's mass spread is **0.095** — roughly 9× tighter than top-k (0.824) or min-p (0.863). **C1 ✅** and **C3 ✅** (the ordering top-p < top-k < min-p holds, though min-p and top-k are swapped versus the prediction). **C2 ✅**: top-p's `n_kept` runs 1 → 12,097, a **12,097× swing** versus the synthetic 355×.

In [2]:
R = {nm: [score_cut(P[t], OPS[nm](P[t], DEFAULT[nm])) for t in range(T)] for nm in OPS}

print(f"{'op':6s} {'metric':9s} {'median':>10} {'min':>10} {'max':>10} {'spread':>10}")
for nm in OPS:
    for key in ["n_kept", "mass_kept", "kl", "expH"]:
        v = np.array([r[key] for r in R[nm]], dtype=float)
        print(f"{nm:6s} {key:9s} {np.median(v):10.3f} {v.min():10.3f} {v.max():10.3f} {v.max()-v.min():10.3f}")

print(f"\n{'op':6s} {'mass spread':>12} {'n_kept range':>22}")
for nm in OPS:
    mk = np.array([r["mass_kept"] for r in R[nm]]); nk = np.array([r["n_kept"] for r in R[nm]])
    print(f"{nm:6s} {mk.max()-mk.min():12.4f} {f'{nk.min()}-{nk.max()} ({nk.max()/max(nk.min(),1):.1f}x)':>22}")

plt.figure(figsize=(6.5, 5.5))
for nm, c in zip(OPS, ["tab:blue", "tab:orange", "tab:green"]):
    plt.loglog(k90_all.numpy(), [r["n_kept"] for r in R[nm]], "o", color=c, alpha=0.6, label=nm)
lim = [1, float(k90_all.max())]
plt.loglog(lim, lim, "k--", lw=1, label="n_kept = k90")
plt.xlabel("position's pre-cut k90"); plt.ylabel("n_kept after truncation")
plt.title("what each default keeps, against what the position actually needs")
plt.legend(fontsize=8); plt.grid(alpha=0.3, which="both")
plt.tight_layout(); plt.show(); plt.close("all")

op     metric        median        min        max     spread
topk   n_kept        50.000     50.000     50.000      0.000
topk   mass_kept      0.783      0.174      0.999      0.824
topk   kl             0.245      0.001      1.746      1.745
topk   expH          14.673      1.037     34.719     33.682
topp   n_kept       222.000      1.000  12097.000  12096.000
topp   mass_kept      0.900      0.900      0.995      0.095
topp   kl             0.105      0.005      0.105      0.100
topp   expH          31.288      1.000   3485.006   3484.006
minp   n_kept         5.000      1.000     24.000     23.000
minp   mass_kept      0.555      0.132      0.995      0.863
minp   kl             0.589      0.005      2.028      2.023
minp   expH           4.009      1.000     18.248     17.248

op      mass spread           n_kept range
topk         0.8244           50-50 (1.0x)
topp         0.0949     1-12097 (12097.0x)
minp         0.8634           1-24 (24.0x)

## Where each operator cuts, position by position

**Why:** spread says how much variation exists; this says *where it comes from*. If `n_kept` rises when the position genuinely widens, the operator is adapting.

**What the run shows.** Top-p's Spearman is **+1.000** — it widens exactly where the distribution widens. Min-p's is **+0.457** — weakly adaptive, and *positive*, not negative as predicted. **C4 fails:** min-p does not become more aggressive at high-entropy positions on real data the way it did on synthetic flat distributions. At the widest position, top-k keeps 50 tokens (0.10% of V), top-p keeps 12,097 (24.6%), min-p keeps 24 (0.05%). **C6 ✅** — all under 25%.

In [3]:
def spearman(a, b):
    if np.std(a) == 0 or np.std(b) == 0:
        return float("nan")
    ra = np.argsort(np.argsort(a)).astype(float)
    rb = np.argsort(np.argsort(b)).astype(float)
    return float(np.corrcoef(ra, rb)[0, 1])

k90np = k90_all.numpy().astype(float)
for nm in OPS:
    nk = np.array([r["n_kept"] for r in R[nm]], dtype=float)
    r = spearman(k90np, nk)
    print(f"{nm:6s} spearman(k90, n_kept) = "
          f"{'constant by definition' if np.isnan(r) else f'{r:+.4f}'}")

wid = int(k90_all.argmax())
print(f"\nwidest position {wid}: k90 = {int(k90_all[wid])}, exp(H) = {float(H_all[wid].exp()):.1f}")
for nm in OPS:
    r = score_cut(P[wid], OPS[nm](P[wid], DEFAULT[nm]))
    print(f"  {nm:6s} n_kept {r['n_kept']:6d} ({r['n_kept']/V*100:5.2f}% of V)  mass {r['mass_kept']:.4f}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for nm, c in zip(OPS, ["tab:blue", "tab:orange", "tab:green"]):
    ax[0].semilogx(k90_all.numpy(), [r["expH"] for r in R[nm]], "o", color=c, alpha=0.6, label=nm)
    ax[1].hist([r["kl"] for r in R[nm]], bins=25, alpha=0.55, color=c, label=nm)
ax[0].semilogx(k90_all.numpy(), H_all.exp().numpy(), "k.", ms=3, alpha=0.5, label="pre-cut exp(H)")
ax[0].set_xlabel("position's k90"); ax[0].set_ylabel("exp(H') after cut")
ax[0].set_title("branching factor after truncation"); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
ax[1].set_xlabel("D_KL(p'||p) = -log(mass_kept)"); ax[1].set_ylabel("positions")
ax[1].set_title("how far each operator moves the distribution"); ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show(); plt.close("all")

topk   spearman(k90, n_kept) = constant by definition
topp   spearman(k90, n_kept) = +1.0000
minp   spearman(k90, n_kept) = +0.4577

widest position 0: k90 = 12097, exp(H) = 5691.7
  topk   n_kept     50 ( 0.10% of V)  mass 0.1745
  topp   n_kept  12097 (24.61% of V)  mass 0.9000
  minp   n_kept     24 ( 0.05% of V)  mass 0.1315

## Temperature and truncation do not commute

**Why:** every serving stack applies both, and the order is a configuration detail rarely stated. It is not a detail.

**The algebra.** For top-k, temperature is rank-preserving (`log(p_i/p_j)` scales by `1/T` but never flips sign), so both orders select the *same* 50 tokens — a definitional prediction that doubles as an implementation check. For top-p and min-p, flattening the curve (`T > 1`) forces a mass threshold further down the ranking, so even the support changes.

**Distance measured.** Total variation, `TVD(a,b) = ½Σ|a_i − b_i|`, bounded in `[0,1]`. KL would explode to infinity when one order keeps a token the other zeroed.

**What the run shows.** Top-k: **0/83** supports differ at any T — rank invariance confirmed. Top-p and min-p: at `T = 1.5`, the majority of positions change support (top-p mean TVD 0.416). **C5 ❌** — at high T the ordering effect exceeds the smallest between-operator difference; ordering matters *more* than which operator you pick.

In [4]:
def tvd(a, b):
    return float(0.5 * (a - b).abs().sum())

print(f"{'T':>5} {'op':6s} {'supp differ':>12} {'mean TVD':>10} {'max TVD':>9}")
for Tt in [0.7, 1.0, 1.5]:
    for nm in OPS:
        diff, tvs = 0, []
        for t in range(T):
            pT = torch.softmax(Z[t] / Tt, dim=-1)
            mA = OPS[nm](pT, DEFAULT[nm]); pA = renorm(pT, mA)
            mB = OPS[nm](P[t], DEFAULT[nm])
            lg = torch.where(mB, Z[t] / Tt, torch.full_like(Z[t], -float("inf")))
            pB = torch.softmax(lg, dim=-1)
            if not bool((mA == mB).all()): diff += 1
            tvs.append(tvd(pA, pB))
        print(f"{Tt:5.1f} {nm:6s} {f'{diff}/{T}':>12} {np.mean(tvs):10.4f} {np.max(tvs):9.4f}")

print(f"\nbetween-operator TVD at T = 1.0 (mean over {T} positions):")
pairs = []
for a in OPS:
    for b in OPS:
        if a >= b: continue
        v = [tvd(renorm(P[t], OPS[a](P[t], DEFAULT[a])), renorm(P[t], OPS[b](P[t], DEFAULT[b]))) for t in range(T)]
        pairs.append(np.mean(v))
        print(f"  {a:5s} vs {b:5s}: mean {np.mean(v):.4f}  max {np.max(v):.4f}")

mk = {nm: np.array([r["mass_kept"] for r in R[nm]]) for nm in OPS}
nk = {nm: np.array([r["n_kept"] for r in R[nm]]) for nm in OPS}
sp = {nm: mk[nm].max() - mk[nm].min() for nm in OPS}
wid_ok = all(score_cut(P[wid], OPS[nm](P[wid], DEFAULT[nm]))["n_kept"] < 0.25*V for nm in OPS)

print(f"\nC1 topk mass spread > 0.5      : {sp['topk']:.4f} -> {'PASS' if sp['topk']>0.5 else 'FAIL'}")
print(f"C2 topp n_kept spread > 355x   : {nk['topp'].max()/max(nk['topp'].min(),1):.1f}x -> "
      f"{'PASS' if nk['topp'].max()/max(nk['topp'].min(),1)>355 else 'FAIL'}")
print(f"C3 ordering topp<minp<topk     : {sp['topp']:.4f} / {sp['minp']:.4f} / {sp['topk']:.4f} -> "
      f"{'PASS' if sp['topp']<sp['minp']<sp['topk'] else 'FAIL'}")
print(f"C4 minp negative correlation   : {spearman(k90np, nk['minp'].astype(float)):+.4f} -> "
      f"{'PASS' if spearman(k90np, nk['minp'].astype(float))<0 else 'FAIL'}")
print(f"C6 all keep < 25% of V at widest: {'PASS' if wid_ok else 'FAIL'}")

    T op      supp differ   mean TVD   max TVD
  0.7 topk           0/83     0.0000    0.0000
  0.7 topp          80/83     0.0778    0.0952
  0.7 minp          68/83     0.0834    0.2464
  1.0 topk           0/83     0.0000    0.0000
  1.0 topp           0/83     0.0000    0.0000
  1.0 minp           0/83     0.0000    0.0000
  1.5 topk           0/83     0.0000    0.0000
  1.5 topp          83/83     0.4160    0.6795
  1.5 minp          77/83     0.2852    0.6887

between-operator TVD at T = 1.0 (mean over 83 positions):
  topk  vs topp : mean 0.1698  max 0.8061
  minp  vs topk : mean 0.2699  max 0.5393
  minp  vs topp : mean 0.3600  max 0.8538

C1 topk mass spread > 0.5      : 0.8244 -> PASS
C2 topp n_kept spread > 355x   : 12097.0x -> PASS
C3 ordering topp<minp<topk     : 0.0949 / 0.8634 / 0.8244 -> FAIL
C4 minp negative correlation   : +0.4577 -> FAIL
C6 all keep < 25% of V at widest: PASS

## Findings

| # | Claim | Predicted | Measured | Verdict |
|---|---|---|---|---|
| C1 | Fixed `k = 50` varies across positions | `mass_kept` spans > 0.5 | **0.8244** (0.174 → 0.999) | ✅ Holds |
| C2 | Top-p's count varies more than synthetic | `n_kept` spread > 355× | **12,097×** (1 → 12,097) | ✅ Holds |
| C3 | Controlled ordering transfers | top-p < min-p < top-k | top-p 0.095 < **top-k 0.824** < min-p 0.863 | ❌ Reversed |
| C4 | Min-p most aggressive at high entropy | negative correlation | Spearman **+0.457** | ❌ Reversed |
| C5 | Ordering matters less than operator choice | ordering TVD < between-op TVD | ordering 0.416 > min between-op 0.170 | ❌ Reversed |
| C6 | All keep < 25% of V at widest position | each keeps < 25% | top-k 0.10%, top-p 24.6%, min-p 0.05% | ✅ Holds |

**Secondary measurements.** Top-p's Spearman(k90, n_kept) = **+1.000** — perfect adaptivity by count. Top-k's rank invariance confirmed: **0/83** supports differ across temperatures. At `T = 1.5` top-p's support differs at **all 83** positions between the two orderings (mean TVD 0.416).

## Discussion and verdict

**The claim in one line.** On real logits the mass-based rule is still the most stable across positions, but it is not adaptive by count — its `n_kept` swings 12,097× over one document — and three of the six pre-registered predictions reverse, two of them sharply.

**What transferred and what did not.** Top-p's stability transferred cleanly: mass spread 0.095, roughly 9× tighter than the alternatives, and its `n_kept` tracks the position's width with Spearman +1.000. But the *interpretation* flips: the previous study read top-p's count-swing as a defect (355× on synthetic shapes); here the swing is **12,097×** and the coefficient is +1.000, which means the swing is the point — top-p is the only operator whose `n_kept` actually matches what each position needs. The defect is the other two, whose counts are pinned or nearly so (min-p never exceeds 24 tokens on distributions whose median `k90` is 222).

**C4 — the synthetic reversal was specific to flat distributions.** Min-p's negative correlation appeared in the controlled study on flat Zipf distributions; on real positions it is **+0.457**. Real high-entropy positions are not Zipf-flat, and min-p's relative-height rule tracks them weakly rather than cutting against them. The previous finding is bounded to its construction.

**C5 — ordering is not a detail at high T.** At `T = 1.5` the temperature-then-cut versus cut-then-temperature supports differ at all 83 positions (mean TVD 0.416), exceeding the smallest between-operator distance (0.170). A configuration switch nobody documents is larger than choosing a different operator. Top-k is the exception — rank invariance holds at every T, which is itself a structural property worth knowing.

**Honesty note.** Eighty-three positions from one document, computed from cached float32 logits promoted to float64. No ground truth: nothing here evaluates whether a cut was *correct*, only how much it varied and how far it moved the distribution.

**Where the next study goes.** Whether distribution shape itself — and therefore the meaning of every default measured here — travels across model scale.

## References

1. Holtzman, A., Buys, J., Du, L., Forbes, M., Choi, Y. (2020). *The Curious Case of Neural Text Degeneration*. arXiv:1904.09751.
2. Fan, A., Lewis, M., Dauphin, Y. (2018). *Hierarchical Neural Story Generation*. arXiv:1805.04833.
3. Nguyen, M., Baker, A., Neo, C., et al. (2024). *Turning Up the Heat: Min-p Sampling*. arXiv:2407.01082.
4. Meister, C., Pimentel, T., Wiher, G., Cotterell, R. (2023). *Locally Typical Sampling*. arXiv:2202.00666.
5. Zhu, D., et al. (2024). *Hot or Cold? Adaptive Temperature Sampling for Code Generation*. AAAI.
6. Allal, L. B., Lozhkov, A., et al. (2025). *SmolLM2: When Smol Goes Big*. arXiv:2502.02737.
7. Cover, T. M., Thomas, J. A. (2006). *Elements of Information Theory*, 2nd ed., Wiley.